# Quant research panel inspection

This notebook explores the Week 10 research outputs:

- local Parquet datasets under `data/dwd`, `data/dws`, and `data/ads`
- optional BigQuery tables
- optional GCS prefixes

Run it from the repository root. Most cells use local Parquet first; BigQuery and GCS cells are disabled by default.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display

from quant_platform.research.config import (
    load_market_context_config,
    load_research_panel_config,
)


pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

research_config = load_research_panel_config("configs/research_panel.yml")
market_config = load_market_context_config("configs/market_context.yml")

universe_name = str(
    research_config.feature_scope.get(
        "universe_name",
        research_config.default_universe_name,
    )
)

print("project_root:", PROJECT_ROOT)
print("factor_set:", research_config.factor_set)
print("label_set:", research_config.label_set)
print("universe_name:", universe_name)
print("context_set:", market_config.context_set)
print("label_horizons:", research_config.label_horizons)
print("rolling return windows:", research_config.rolling_windows["returns"])


In [ ]:
def parquet_files(root: str | Path) -> list[Path]:
    root_path = Path(root)
    return sorted(root_path.rglob("*.parquet"))


def parquet_glob(root: str | Path) -> str:
    return (Path(root) / "**" / "*.parquet").as_posix()


def show_df(
    df: pd.DataFrame,
    rows: int = 20,
    title: str | None = None,
) -> None:
    if title:
        display(Markdown(f"### {title}"))
    display(df.head(rows))


def dataset_summary(
    name: str,
    root: str | Path,
    *,
    date_column: str | None = "date",
) -> pd.DataFrame:
    files = parquet_files(root)
    record = {
        "dataset": name,
        "root": str(root),
        "file_count": len(files),
        "row_count": np.nan,
        "ticker_count": np.nan,
        "security_id_count": np.nan,
        "min_date": None,
        "max_date": None,
    }

    if not files:
        return pd.DataFrame([record])

    con = duckdb.connect()
    try:
        summary_exprs = ["COUNT(*) AS row_count"]

        if date_column is not None:
            summary_exprs.extend(
                [
                    f"MIN(CAST({date_column} AS DATE)) AS min_date",
                    f"MAX(CAST({date_column} AS DATE)) AS max_date",
                ]
            )

        sample = pd.read_parquet(files[0])
        if "ticker" in sample.columns:
            summary_exprs.append("COUNT(DISTINCT ticker) AS ticker_count")
        if "security_id" in sample.columns:
            summary_exprs.append(
                "COUNT(DISTINCT security_id) AS security_id_count"
            )

        sql = f"""
        SELECT
          {", ".join(summary_exprs)}
        FROM read_parquet(?)
        """

        result = con.execute(sql, [parquet_glob(root)]).fetchdf().iloc[0]
    finally:
        con.close()

    for column in result.index:
        record[column] = result[column]

    return pd.DataFrame([record])


In [ ]:
paths = {
    "dim_market_context_symbol": (
        Path("data/dwd/security_master/dim_market_context_symbol")
    ),
    "dwd_market_context_price_daily": (
        Path(market_config.price_dwd_root)
        / f"context_set={market_config.context_set}"
    ),
    "dws_market_context_features_daily": (
        Path(research_config.market_context_feature_root)
        / f"context_set={market_config.context_set}"
    ),
    "dws_equity_features_daily": (
        Path(research_config.feature_output_root)
        / f"factor_set={research_config.factor_set}"
    ),
    "dws_equity_forward_returns_daily": (
        Path(research_config.label_output_root)
        / f"label_set={research_config.label_set}"
    ),
    "ads_equity_research_panel_daily": (
        Path(research_config.panel_output_root)
        / f"universe_name={universe_name}"
        / f"factor_set={research_config.factor_set}"
    ),
}

summaries = []

for name, root in paths.items():
    date_column = None if name == "dim_market_context_symbol" else "date"
    summaries.append(
        dataset_summary(
            name,
            root,
            date_column=date_column,
        )
    )

inventory = pd.concat(summaries, ignore_index=True)
show_df(inventory, 20, "Local dataset inventory")


In [ ]:
panel_root = paths["ads_equity_research_panel_daily"]
panel_files = parquet_files(panel_root)

if not panel_files:
    raise FileNotFoundError(f"No panel Parquet files found under {panel_root}")

sample_panel = pd.read_parquet(panel_files[-1])

column_families = {
    "identity": [
        "date",
        "universe_name",
        "membership_month",
        "security_id",
        "ticker",
        "rank",
    ],
    "returns": [c for c in sample_panel.columns if c.startswith("ret_")],
    "momentum": [c for c in sample_panel.columns if c.startswith("mom_")],
    "reversal": [c for c in sample_panel.columns if c.startswith("rev_")],
    "technical": [c for c in sample_panel.columns if c.startswith("tech_")],
    "candlestick": [c for c in sample_panel.columns if c.startswith("cdl_")],
    "labels": [c for c in sample_panel.columns if c.startswith("label_fwd_ret_")],
    "market_context": [c for c in sample_panel.columns if c.startswith("mkt_")],
    "excess": [c for c in sample_panel.columns if c.startswith("excess_")],
}

family_summary = pd.DataFrame(
    [
        {
            "family": family,
            "column_count": len(columns),
            "sample_columns": ", ".join(columns[:10]),
        }
        for family, columns in column_families.items()
    ]
)

show_df(family_summary, 20, "Panel column-family summary")
print("total panel columns:", len(sample_panel.columns))


In [ ]:
con = duckdb.connect()
try:
    month_summary = con.execute(
        """
        SELECT
          DATE_TRUNC('month', CAST(date AS DATE)) AS month,
          COUNT(*) AS row_count,
          COUNT(DISTINCT date) AS trading_days,
          COUNT(DISTINCT security_id) AS security_ids,
          MIN(rank) AS min_rank,
          MAX(rank) AS max_rank
        FROM read_parquet(?)
        GROUP BY 1
        ORDER BY 1
        """,
        [parquet_glob(panel_root)],
    ).fetchdf()

    latest_date = con.execute(
        """
        SELECT MAX(CAST(date AS DATE)) AS latest_date
        FROM read_parquet(?)
        """,
        [parquet_glob(panel_root)],
    ).fetchdf()["latest_date"].iloc[0]

    latest_sample = con.execute(
        """
        SELECT
          date,
          rank,
          ticker,
          security_id,
          ret_21d,
          tech_rsi_14,
          tech_rsi_21,
          label_fwd_ret_21d,
          mkt_spy_ret_21d,
          excess_ret_21d_vs_spy
        FROM read_parquet(?)
        WHERE CAST(date AS DATE) = ?
        ORDER BY rank
        LIMIT 50
        """,
        [parquet_glob(panel_root), latest_date],
    ).fetchdf()
finally:
    con.close()

show_df(month_summary.tail(24), 24, "Panel monthly coverage")
print("latest panel date:", latest_date)
show_df(latest_sample, 50, "Latest-date top-ranked sample")


In [ ]:
label_cols = [
    f"label_fwd_ret_{horizon}d"
    for horizon in research_config.label_horizons
]
has_label_cols = [
    f"has_label_fwd_ret_{horizon}d"
    for horizon in research_config.label_horizons
]

select_exprs = ["COUNT(*) AS row_count"]

for label_col in label_cols:
    select_exprs.append(
        f"SUM(CASE WHEN {label_col} IS NOT NULL THEN 1 ELSE 0 END) "
        f"AS {label_col}_non_null"
    )

for has_col in has_label_cols:
    select_exprs.append(
        f"SUM(CASE WHEN {has_col} THEN 1 ELSE 0 END) AS {has_col}_true"
    )

con = duckdb.connect()
try:
    label_coverage = con.execute(
        f"""
        SELECT
          {", ".join(select_exprs)}
        FROM read_parquet(?)
        """,
        [parquet_glob(panel_root)],
    ).fetchdf()
finally:
    con.close()

show_df(label_coverage, 5, "Panel label coverage")


In [ ]:
feature_candidates = [
    "ret_1d",
    "ret_2d",
    "ret_5d",
    "ret_10d",
    "ret_21d",
    "ret_63d",
    "mom_21d",
    "mom_63d",
    "mom_126d",
    "mom_252_21d",
    "rev_1d",
    "rev_2d",
    "rev_5d",
    "realized_vol_21d",
    "price_position_63d",
    "price_position_252d",
    "sma_20_ratio",
    "sma_50_ratio",
    "sma_200_ratio",
    "tech_rsi_14",
    "tech_rsi_21",
    "tech_mfi_14",
    "tech_mfi_21",
    "tech_adx_14",
    "tech_adx_21",
    "tech_cmo_14",
    "tech_cmo_21",
    "tech_bb_position_20",
    "tech_tema_20_ratio",
    "tech_tema_50_ratio",
    "mkt_spy_ret_21d",
    "mkt_qqq_ret_21d",
    "excess_ret_21d_vs_spy",
]

available_features = [
    column for column in feature_candidates if column in sample_panel.columns
]

con = duckdb.connect()
try:
    missing_exprs = [
        f"AVG(CASE WHEN {column} IS NULL THEN 1 ELSE 0 END) AS {column}"
        for column in available_features
    ]
    missingness_raw = con.execute(
        f"""
        SELECT
          {", ".join(missing_exprs)}
        FROM read_parquet(?)
        """,
        [parquet_glob(panel_root)],
    ).fetchdf()
finally:
    con.close()

missingness = (
    missingness_raw.T.reset_index()
    .rename(columns={"index": "feature", 0: "missing_ratio"})
    .sort_values("missing_ratio", ascending=False)
)

show_df(missingness, 100, "Selected feature missingness")


In [ ]:
def recent_panel_frame(
    *,
    months_back: int = 24,
    columns: list[str] | None = None,
) -> pd.DataFrame:
    files = parquet_files(panel_root)

    if not files:
        raise FileNotFoundError(f"No panel files under {panel_root}")

    month_paths = []
    for path in files:
        year = None
        month = None

        for part in path.parts:
            if part.startswith("year="):
                year = int(part.removeprefix("year="))
            elif part.startswith("month="):
                month = int(part.removeprefix("month="))

        if year is not None and month is not None:
            month_paths.append((pd.Period(f"{year}-{month:02d}"), path))

    if not month_paths:
        raise ValueError("Could not parse panel year/month partitions")

    latest_month = max(period for period, _ in month_paths)
    start_month = latest_month - months_back + 1

    selected = [
        path
        for period, path in month_paths
        if start_month <= period <= latest_month
    ]

    return pd.concat(
        [pd.read_parquet(path, columns=columns) for path in selected],
        ignore_index=True,
    )


ic_features = [
    column for column in available_features if column in sample_panel.columns
]
target = "label_fwd_ret_21d"

analysis_columns = [
    "date",
    "ticker",
    "security_id",
    target,
    *ic_features,
]

analysis_columns = [
    column for column in analysis_columns if column in sample_panel.columns
]

panel_recent = recent_panel_frame(
    months_back=24,
    columns=analysis_columns,
)
panel_recent["date"] = pd.to_datetime(panel_recent["date"]).dt.date

print("recent rows:", len(panel_recent))
print("recent min date:", panel_recent["date"].min())
print("recent max date:", panel_recent["date"].max())
print("target:", target)


In [ ]:
ic_rows = []

for factor in ic_features:
    if factor not in panel_recent.columns:
        continue

    by_date = (
        panel_recent[["date", factor, target]]
        .dropna()
        .groupby("date")
        .apply(
            lambda group: group[factor].corr(
                group[target],
                method="spearman",
            ),
            include_groups=False,
        )
        .dropna()
    )

    if by_date.empty:
        continue

    ic_rows.append(
        {
            "factor": factor,
            "date_count": len(by_date),
            "mean_ic": by_date.mean(),
            "median_ic": by_date.median(),
            "std_ic": by_date.std(),
            "positive_ic_ratio": (by_date > 0).mean(),
        }
    )

ic_summary = pd.DataFrame(ic_rows)

if not ic_summary.empty:
    ic_summary["ic_ir"] = ic_summary["mean_ic"] / ic_summary["std_ic"]
    ic_summary = ic_summary.sort_values("mean_ic", ascending=False)

show_df(ic_summary, 100, "Spearman IC summary over recent panel")


In [ ]:
FACTOR_FOR_DECILES = "mom_252_21d"
LABEL_FOR_DECILES = "label_fwd_ret_21d"

decile_input = panel_recent[
    ["date", "ticker", FACTOR_FOR_DECILES, LABEL_FOR_DECILES]
].dropna()


def assign_decile(series: pd.Series) -> pd.Series:
    if series.nunique(dropna=True) < 10:
        return pd.Series(np.nan, index=series.index)

    return pd.qcut(
        series,
        q=10,
        labels=False,
        duplicates="drop",
    ) + 1


decile_input = decile_input.copy()
decile_input["decile"] = (
    decile_input.groupby("date")[FACTOR_FOR_DECILES]
    .transform(assign_decile)
)

decile_summary = (
    decile_input.dropna(subset=["decile"])
    .groupby("decile")
    .agg(
        row_count=(LABEL_FOR_DECILES, "size"),
        avg_forward_return=(LABEL_FOR_DECILES, "mean"),
        median_forward_return=(LABEL_FOR_DECILES, "median"),
    )
    .reset_index()
    .sort_values("decile")
)

show_df(decile_summary, 20, f"Decile analysis: {FACTOR_FOR_DECILES}")


In [ ]:
market_feature_root = paths["dws_market_context_features_daily"]
market_files = parquet_files(market_feature_root)

if not market_files:
    raise FileNotFoundError(
        f"No market context feature files found under {market_feature_root}"
    )

con = duckdb.connect()
try:
    market_summary = con.execute(
        """
        SELECT
          context_group,
          ticker,
          COUNT(*) AS rows,
          MIN(CAST(date AS DATE)) AS min_date,
          MAX(CAST(date AS DATE)) AS max_date
        FROM read_parquet(?)
        GROUP BY 1, 2
        ORDER BY 1, 2
        """,
        [parquet_glob(market_feature_root)],
    ).fetchdf()

    latest_market = con.execute(
        """
        SELECT
          date,
          context_group,
          ticker,
          ret_1d,
          ret_21d,
          tech_rsi_14,
          tech_rsi_21,
          tech_adx_14,
          tech_adx_21
        FROM read_parquet(?)
        WHERE CAST(date AS DATE) = (
          SELECT MAX(CAST(date AS DATE))
          FROM read_parquet(?)
        )
        ORDER BY context_group, ticker
        """,
        [parquet_glob(market_feature_root), parquet_glob(market_feature_root)],
    ).fetchdf()
finally:
    con.close()

show_df(market_summary, 50, "Market context feature coverage")
show_df(latest_market, 50, "Latest market context features")


## Optional BigQuery inspection

Set `RUN_BIGQUERY = True` after research outputs have been published to BigQuery.


In [ ]:
RUN_BIGQUERY = False

if RUN_BIGQUERY:
    from google.cloud import bigquery

    project_id = os.environ["GCP_PROJECT_ID"]
    dataset_id = os.environ["BIGQUERY_DWH_DATASET"]
    location = os.getenv("GCP_LOCATION", "US")

    client = bigquery.Client(project=project_id, location=location)

    bq_tables = [
        "dim_market_context_symbol",
        "dwd_market_context_price_daily",
        "dws_market_context_features_daily",
        "dws_equity_features_daily",
        "dws_equity_forward_returns_daily",
        "ads_equity_research_panel_daily",
    ]

    table_rows = []

    for table_name in bq_tables:
        table_id = f"{project_id}.{dataset_id}.{table_name}"
        table = client.get_table(table_id)
        table_rows.append(
            {
                "table_name": table_name,
                "rows": table.num_rows,
                "schema_fields": len(table.schema),
                "partition_field": (
                    table.time_partitioning.field
                    if table.time_partitioning is not None
                    else None
                ),
                "clustering_fields": table.clustering_fields,
            }
        )

    show_df(pd.DataFrame(table_rows), 20, "BigQuery research tables")


In [ ]:
if RUN_BIGQUERY:
    project_id = os.environ["GCP_PROJECT_ID"]
    dataset_id = os.environ["BIGQUERY_DWH_DATASET"]
    location = os.getenv("GCP_LOCATION", "US")

    client = bigquery.Client(project=project_id, location=location)
    table = f"`{project_id}.{dataset_id}.ads_equity_research_panel_daily`"

    bq_summary_sql = f"""
    SELECT
      COUNT(*) AS row_count,
      COUNT(DISTINCT date) AS date_count,
      COUNT(DISTINCT security_id) AS security_id_count,
      COUNT(DISTINCT ticker) AS ticker_count,
      MIN(date) AS min_date,
      MAX(date) AS max_date
    FROM {table}
    """

    bq_summary = client.query(
        bq_summary_sql,
        location=location,
    ).to_dataframe()

    show_df(bq_summary, 5, "BigQuery panel summary")


## Optional GCS inspection

Set `RUN_GCS = True` after local outputs have been synced to GCS.


In [ ]:
RUN_GCS = False

if RUN_GCS:
    from google.cloud import storage

    project_id = os.environ["GCP_PROJECT_ID"]
    bucket_name = os.environ["GCS_BUCKET"]

    storage_client = storage.Client(project=project_id)
    bucket = storage_client.bucket(bucket_name)

    prefixes = [
        "dwd/security_master/dim_market_context_symbol/",
        "dwd/market_context_price_daily/",
        "dws/market_context_features_daily/",
        "dws/equity_features_daily/",
        "dws/equity_forward_returns_daily/",
        "ads/equity_research_panel_daily/",
    ]

    rows = []

    for prefix in prefixes:
        blobs = list(bucket.list_blobs(prefix=prefix, max_results=20))
        rows.append(
            {
                "prefix": prefix,
                "sample_object_count": len(blobs),
                "sample_objects": ", ".join(blob.name for blob in blobs[:3]),
            }
        )

    show_df(pd.DataFrame(rows), 20, "GCS research output prefixes")
